# 52 · The QR codes with Fieldwork's mark, checked against OpenCV

## What this checks

A worked example's write-up carries QR codes. Since Fieldwork release 0.114.0, each code has Fieldwork's mark in
the middle. This check covers the codes Fieldwork release 0.200.0 prints. It checks no widget: the codes belong to
the write-up.

To make room for the mark, the squares in the middle of each code are cleared. The code is drawn at error
correction H, the highest level. Error correction is the spare data that lets a damaged or covered code still be
read. A mark that stops a code being read sends a reader nowhere. So it is worth checking.

## How it is checked

**OpenCV**, a widely used library for reading images, reads each code. It has two detectors, the classic one and a
newer one, and each tries four sizes. Its authors did not write Fieldwork's code. No code is shared with Fieldwork.

Each code is read twice:

- **Patch light**, as Fieldwork draws it.
- **Patch dark**, as a camera might see a dark green mark.

A second encoder, **segno**, draws its own code for each address. It uses the same level and size, with no mark. This
is a control on the reader. If OpenCV cannot read segno's code either, a miss is the reader's limit, not the mark's.

The data are 28 addresses: every address Fieldwork would print, with each code's squares as drawn and where the mark
sits.

OpenCV is not in the browser kernel, so this check runs in CPython only. In the browser, the notebook says so and
skips the rest.

## What should happen

1. **Every marked code reads back** to its address, exactly, by at least one of OpenCV's detectors. This must hold
   with the patch light and with it dark.
2. **No miss is the mark's.** Where a detector misses a marked code, it must miss segno's unmarked code too.

These criteria were fixed before the check was run.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
try:
    from checks.check_52_qr_mark import run
except ImportError as missing:
    report = None
    print("Not runnable here:", missing)
else:
    report = run(Path.cwd() / "fixtures" / "qr-mark-001.json")
    print("agrees:", report["agrees"])
    print("read:", report["read"])
    print("differs:", report["differs"])

## The result on record

First recorded on 14 November 2026, against Fieldwork release 0.114.0. Rerun on 26 November 2026, against Fieldwork
release 0.200.0: **Agrees**. Fieldwork's record of widget checks does not list this check, because the codes belong
to no widget.

All 28 marked codes read back to their addresses, with the patch light and with it dark. The ArUco-based detector
read all 28, and all 28 of segno's codes. The classic detector read 27 marked codes and 26 of segno's. Where it
missed a marked code, it missed segno's unmarked code too, so no miss was the mark's.

Running the cell above repeats the check on the same data.

## What this does not show

- That a phone camera reads a printed code with the mark, at a given size and light.
- That every reader app copes with the mark as OpenCV does.
- Anything in the browser kernel. OpenCV is not there, so this ran in CPython only.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    for item in report["notEstablished"]:
        print("-", item)